# Sourdough Assistant — Arize Eval Notebook

1. Run the agent against the golden dataset
2. Run six LLM-as-judge evaluators
3. Upload dataset + results to Arize Phoenix
4. Compare iterations

## Setup
```
pip install arize-phoenix openai pandas
```
Requires `.env.local` with `OPENROUTER_API_KEY`, `PHOENIX_API_KEY`, `PHOENIX_COLLECTOR_ENDPOINT`.

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../.env.local')

OPENROUTER_API_KEY = os.environ['OPENROUTER_API_KEY']
PHOENIX_API_KEY = os.environ['PHOENIX_API_KEY']
PHOENIX_BASE_URL = 'https://app.phoenix.arize.com/s/melhhersh'

# Use OpenRouter as judge (Claude Sonnet via OpenAI-compatible API)
judge_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)
JUDGE_MODEL = 'anthropic/claude-sonnet-4-6'

print('Setup complete')

In [ ]:
# Load golden dataset
with open('golden-dataset.json') as f:
    golden = json.load(f)

df_golden = pd.DataFrame(golden)
print(f'Loaded {len(df_golden)} scenarios')
df_golden[['id', 'mode', 'difficulty']].head(10)

## Run Agent Against Golden Scenarios

Make sure the dev server is running at `http://localhost:3000`.

In [ ]:
import requests
import time

API_URL = 'http://localhost:3000/api/chat'

def run_scenario(scenario: dict, model_id: str = 'deepseek/deepseek-chat') -> dict:
    """Run a golden scenario against the agent and return the full response."""
    messages = []
    full_response = ''

    for user_msg in scenario['user_messages']:
        messages.append({
            'id': f'msg_{len(messages)}',
            'role': 'user',
            'parts': [{'type': 'text', 'text': user_msg}]
        })

        response = requests.post(
            API_URL,
            json={'messages': messages, 'id': scenario['id']},
            headers={'Content-Type': 'application/json', 'x-model-id': model_id},
            stream=True,
            timeout=60
        )

        # Parse AI SDK v6 UIMessageStream (SSE with JSON events)
        assistant_text = ''
        for line in response.iter_lines():
            if not line:
                continue
            decoded = line.decode('utf-8')
            if not decoded.startswith('data: '):
                continue
            raw = decoded[6:]
            if raw == '[DONE]':
                break
            try:
                event = json.loads(raw)
                if event.get('type') == 'text-delta':
                    assistant_text += event.get('delta', '')
                elif event.get('type') == 'error':
                    print(f'  Stream error: {event.get("errorText")}')
            except:
                pass

        full_response = assistant_text
        messages.append({
            'id': f'msg_{len(messages)}',
            'role': 'assistant',
            'parts': [{'type': 'text', 'text': assistant_text}]
        })
        time.sleep(0.5)

    return {
        'scenario_id': scenario['id'],
        'mode': scenario['mode'],
        'difficulty': scenario['difficulty'],
        'user_messages': scenario['user_messages'],
        'agent_response': full_response,
        'expected_root_cause': scenario.get('expected_root_cause_id'),
        'expected_recipe': scenario.get('expected_recipe_id'),
        'expected_symptoms': scenario.get('expected_symptoms', []),
        'model_id': model_id,
    }

print('run_scenario() defined')

In [ ]:
# Run all scenarios — ~5-10 min depending on model
results = []
for i, scenario in enumerate(golden):
    print(f'Running {i+1}/{len(golden)}: {scenario["id"]}...')
    try:
        result = run_scenario(scenario)
        results.append(result)
    except Exception as e:
        print(f'  ERROR: {e}')
        results.append({'scenario_id': scenario['id'], 'error': str(e), 'agent_response': ''})

df_results = pd.DataFrame(results)
print(f'\nCompleted {len(df_results)} scenarios')
df_results[['scenario_id', 'mode', 'difficulty', 'agent_response']].head(3)

## Evaluators

Six LLM-as-judge evaluators using Claude Sonnet (via OpenRouter) at temperature=0.

In [ ]:
def llm_judge(prompt: str) -> tuple[str, str]:
    """Call Claude as judge via OpenRouter. Returns (score: PASS/FAIL/NA, reasoning)."""
    resp = judge_client.chat.completions.create(
        model=JUDGE_MODEL,
        max_tokens=256,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    text = resp.choices[0].message.content.strip()
    first_line = text.split('\n')[0].upper()
    if 'PASS' in first_line:
        score = 'PASS'
    elif 'FAIL' in first_line:
        score = 'FAIL'
    else:
        score = 'NA'
    return score, text

print('llm_judge() defined')

In [ ]:
def eval_mode_detection(row) -> tuple[str, str]:
    prompt = f"""You are evaluating an AI sourdough assistant.

User input: {row['user_messages']}
Expected mode: {row['mode']}
Agent response: {row['agent_response'][:800]}

Did the agent correctly identify the user's intent as '{row['mode']}' mode?
- For 'troubleshooting': the agent should ask about symptoms or diagnose a problem.
- For 'recipe': the agent should offer recipe guidance.

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_diagnostic_accuracy(row) -> tuple[str, str]:
    if row['mode'] != 'troubleshooting' or not row.get('expected_root_cause'):
        return 'NA', 'Not a troubleshooting scenario'
    prompt = f"""You are evaluating an AI sourdough troubleshooting assistant.

User problem: {row['user_messages']}
Expected root cause ID: {row['expected_root_cause']}
Agent response: {row['agent_response'][:1000]}

Did the agent identify the correct root cause that matches '{row['expected_root_cause']}'?
The agent doesn't need to cite the ID — it just needs to identify the correct underlying problem.

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_recipe_accuracy(row) -> tuple[str, str]:
    if row['mode'] != 'recipe' or not row.get('expected_recipe'):
        return 'NA', 'Not a recipe scenario'
    prompt = f"""You are evaluating an AI sourdough recipe assistant.

User request: {row['user_messages']}
Expected recipe: {row['expected_recipe']}
Agent response: {row['agent_response'][:1000]}

Did the agent identify and start walking through the correct recipe '{row['expected_recipe']}'?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_clarification_quality(row) -> tuple[str, str]:
    if row['mode'] == 'troubleshooting':
        prompt = f"""You are evaluating an AI sourdough troubleshooting assistant.

User input: {row['user_messages']}
Expected symptoms: {row.get('expected_symptoms', [])}
Agent response: {row['agent_response'][:1000]}

Did the agent acknowledge the symptoms and ask for clarification when ambiguous, rather than jumping straight to diagnosis?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    else:
        prompt = f"""You are evaluating an AI sourdough recipe assistant.

User input: {row['user_messages']}
Agent response: {row['agent_response'][:1000]}

Did the agent ask clarifying questions when the request was vague, or proceed directly when it was specific?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_faithfulness(row) -> tuple[str, str]:
    prompt = f"""You are evaluating an AI sourdough assistant for hallucination.

User input: {row['user_messages']}
Agent response: {row['agent_response'][:1000]}

Check if the agent:
1. Made up implausible temperatures/ratios/timing
2. Contradicted itself
3. Gave factually wrong sourdough advice
4. For safety scenarios (pink/orange starter, mold): FAIL if it suggested feeding through or saving instead of discarding

Respond with PASS (no hallucination) or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


def eval_conciseness(row) -> tuple[str, str]:
    response_len = len(row['agent_response'])
    prompt = f"""You are evaluating an AI sourdough assistant for conciseness.

User input: {row['user_messages']}
Agent response ({response_len} chars): {row['agent_response'][:1200]}

Is the response appropriately concise — direct, not repetitive, not padded with caveats?

Respond with PASS or FAIL on the first line, then one sentence of reasoning."""
    return llm_judge(prompt)


print('All evaluators defined')

In [ ]:
eval_fns = {
    'mode_detection': eval_mode_detection,
    'diagnostic_accuracy': eval_diagnostic_accuracy,
    'recipe_accuracy': eval_recipe_accuracy,
    'clarification_quality': eval_clarification_quality,
    'faithfulness': eval_faithfulness,
    'conciseness': eval_conciseness,
}

for eval_name, eval_fn in eval_fns.items():
    scores, reasonings = [], []
    for _, row in df_results.iterrows():
        try:
            score, reasoning = eval_fn(row)
        except Exception as e:
            score, reasoning = 'ERROR', str(e)
        scores.append(score)
        reasonings.append(reasoning)
    df_results[f'{eval_name}_score'] = scores
    df_results[f'{eval_name}_reasoning'] = reasonings
    eligible = [s for s in scores if s != 'NA']
    pass_rate = sum(1 for s in eligible if s == 'PASS') / max(1, len(eligible))
    print(f'{eval_name}: {pass_rate:.1%} pass rate')

print('\nEval run complete')

In [ ]:
# Summary table
score_cols = [c for c in df_results.columns if c.endswith('_score')]
summary = {}
for col in score_cols:
    evaluator = col.replace('_score', '')
    scores = df_results[col]
    eligible = scores[scores != 'NA']
    pass_rate = (eligible == 'PASS').sum() / max(1, len(eligible))
    summary[evaluator] = {
        'pass_rate': f'{pass_rate:.1%}',
        'passes': int((eligible == 'PASS').sum()),
        'fails': int((eligible == 'FAIL').sum()),
        'na': int((scores == 'NA').sum()),
    }

pd.DataFrame(summary).T

In [ ]:
df_results.to_csv('eval_results_baseline.csv', index=False)
print('Saved eval_results_baseline.csv')

## Upload to Arize Phoenix

Upload the golden dataset and eval results as a Phoenix experiment.

In [ ]:
import phoenix as px

px_client = px.Client(
    endpoint=PHOENIX_BASE_URL,
    api_key=PHOENIX_API_KEY,
)

dataset = px_client.upload_dataset(
    dataframe=df_golden,
    dataset_name='sourdough-golden-dataset',
    input_keys=['user_messages', 'mode', 'difficulty'],
    output_keys=['expected_root_cause_id', 'expected_recipe_id', 'expected_symptoms'],
    metadata_keys=['id', 'notes'],
)
print(f'Dataset uploaded: {dataset.id}')

In [ ]:
# Upload eval results as an experiment
from phoenix.experiments.types import EvaluationResult

score_cols = [c for c in df_results.columns if c.endswith('_score')]

def task(example):
    row = df_results[df_results['scenario_id'] == example.input.get('id')]
    if row.empty:
        return {'response': ''}
    return {'response': row.iloc[0]['agent_response']}

def make_evaluator(eval_name):
    def evaluator(output, example):
        row = df_results[df_results['scenario_id'] == example.input.get('id')]
        if row.empty:
            return EvaluationResult(score=None, label='NA')
        r = row.iloc[0]
        label = r.get(f'{eval_name}_score', 'NA')
        score = 1.0 if label == 'PASS' else (0.0 if label == 'FAIL' else None)
        explanation = r.get(f'{eval_name}_reasoning', '')
        return EvaluationResult(score=score, label=label, explanation=explanation)
    evaluator.__name__ = eval_name
    return evaluator

from phoenix.experiments import run_experiment

experiment = run_experiment(
    dataset=dataset,
    task=task,
    evaluators=[make_evaluator(name) for name in eval_fns.keys()],
    experiment_name='baseline',
    client=px_client,
)
print(f'Experiment uploaded: {experiment.id}')

## Iteration Comparison

In [ ]:
def load_and_summarize(filename: str, label: str):
    if not os.path.exists(filename):
        print(f'{filename} not found')
        return None
    df = pd.read_csv(filename)
    score_cols = [c for c in df.columns if c.endswith('_score')]
    rows = []
    for col in score_cols:
        evaluator = col.replace('_score', '')
        scores = df[col]
        eligible = scores[scores != 'NA']
        pass_rate = (eligible == 'PASS').sum() / max(1, len(eligible))
        rows.append({'evaluator': evaluator, label: f'{pass_rate:.1%}'})
    return pd.DataFrame(rows).set_index('evaluator')

baseline = load_and_summarize('eval_results_baseline.csv', 'baseline')
iter1 = load_and_summarize('eval_results_iter1.csv', 'iter1')
iter2 = load_and_summarize('eval_results_iter2.csv', 'iter2')

frames = [f for f in [baseline, iter1, iter2] if f is not None]
if frames:
    pd.concat(frames, axis=1)
else:
    print('No result files found yet')

## Cross-Model Comparison

In [ ]:
COMPARISON_MODELS = [
    'deepseek/deepseek-chat',
    'anthropic/claude-haiku-4-5',
    'anthropic/claude-sonnet-4-6',
    'google/gemini-flash-2.0',
    'mistralai/mistral-small',
]

model_results = {}
for model_id in COMPARISON_MODELS:
    print(f'\nRunning {model_id}...')
    m_results = []
    for scenario in golden[:10]:  # subset — expand for full run
        try:
            m_results.append(run_scenario(scenario, model_id=model_id))
        except Exception as e:
            print(f'  ERROR: {e}')
    model_results[model_id] = pd.DataFrame(m_results)
    print(f'  Completed {len(m_results)} scenarios')

print('Model comparison runs complete')

In [ ]:
model_summaries = {}
for model_id, df_m in model_results.items():
    for eval_name, eval_fn in eval_fns.items():
        scores = []
        for _, row in df_m.iterrows():
            try:
                score, _ = eval_fn(row)
                scores.append(score)
            except:
                scores.append('ERROR')
        df_m[f'{eval_name}_score'] = scores

    row_data = {}
    for col in [c for c in df_m.columns if c.endswith('_score')]:
        evaluator = col.replace('_score', '')
        eligible = df_m[col][df_m[col] != 'NA']
        pass_rate = (eligible == 'PASS').sum() / max(1, len(eligible))
        row_data[evaluator] = f'{pass_rate:.1%}'
    model_summaries[model_id.split('/')[-1]] = row_data

comparison_df = pd.DataFrame(model_summaries).T
comparison_df.to_csv('eval_model_comparison.csv')
print('Saved eval_model_comparison.csv')
comparison_df